# Idea of wedding planer

1. Travel Agent 
- searching for flights to the destination of wedding

2. Venue 
- decoration
- searching for a right place

3. DJ
- Creating playlist

4. Chief 
- menu, list of desserts




# Sub-agents
the crue for planning the wedding

In [39]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any
from tavily import TavilyClient
  

tavily_client = TavilyClient()

# Travel Agent
kiwi_client = MultiServerMCPClient({
"travel_server": {
		"transport": "streamable_http",
		"url": "https://mcp.kiwi.com"}})

kiwi_tools = await kiwi_client.get_tools()

class FlightInput(BaseModel):
    destination: str
    departureDate: str
    returnDate: Optional[str] = None  # Now it's optional

travel_agent = create_agent(
    model= "gpt-5-nano",
    tools=kiwi_tools,
    system_prompt="You are a travel agent and all flights proposed by you must contain return fligth. You must return a \
    FlightOutput shceme with destination, deperture and return date, price per one person and link to purchase a ticket",
)

#Venue Agent
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    
    return tavily_client.search(query) 


class Venue(BaseModel):
    name:str
    location: str
    description:str
    capacity: int
    price: float 


venue_agent = create_agent(
    model= "gpt-5-nano",
    tools=[web_search],
    system_prompt="""You are a venue agent and your task is searching the best place for a weeding. 
     You must return a single flat JSON object with exactly these fields:
     - name:str (name of the place)
     - location:str (name of the location [city, country])
     - description(str)
     - capacity:int (maximum amount of people that fit location)
     - price:float (price of renting the location for a wedding time)
     """,
    response_format=Venue,
)


# DJ
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"


class Playlist(BaseModel):
    name:str
    length:float

dj_agent = create_agent(
    model= "gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="""You are a dj agent and you task is to create perfect playlist for the venue.
    You must return a single flat JSON object with exactly these fields:
    - name:string (name of playlist)
    - length:float (length of playlist in minutes)
    
    Do NOT wrap the response in any wrapper.
    """,
    response_format=Playlist,
    
)


#Chief
class Menu(BaseModel):
    name:str
    description:str
    dishes: list[str]
    price_per_capita:float


Chief = create_agent(
    model='gpt-5-nano',
    system_prompt=""""You are a chef advisor. Find the best wedding menu.
    
You MUST return a single flat JSON object with exactly these fields:
- name: string (menu name)
- description: string (short description)  
- dishes: list of strings (list of dish names)
- price_per_capita: float (estimated price per person in PLN)

Do NOT wrap the response in a 'variations' key or any other wrapper.""",
    tools=[web_search],
    response_format=Menu,
)



# Supervisor
and Subagents calling

In [40]:
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import AgentState

@tool
def call_travel_agent(request:str) -> str:
    """Call travel agent to find flight to needed destination"""
    response = travel_agent.invoke({"messages":[HumanMessage(content=request)]})

    return response['messages'][-1].content

@tool
def call_venue_agent(request:str) -> Venue:
    """Call an agent for searching a place of an event"""
    response = venue_agent.invoke({"messages":[HumanMessage(content=request)]})

    return response['messages'][-1].content

@tool
def call_dj_agent(request:str) -> Playlist:
    """Query a dj agent to find a sutiable playlist"""
    response = dj_agent.invoke({"messages":[HumanMessage(content=request)]})

    return response['messages'][-1].content

@tool
def call_chief_agent(request:str) -> Menu:
    """Call and agent to create a Menu for a specific venue"""
    response = Chief.invoke({"messages":[HumanMessage(content=request)]})

    return response['messages'][-1].content



class WeddingState(AgentState):
    name: str
    setting: str
    location: str
    description: str
    persons: int
    price: float
    messages: list 

@tool
def update_wedding_plan(
    name: str,
    setting: str,
    location: str,
    description: str,
    persons: int,
    price: float,
    runtime: ToolRuntime
) -> Command:
    """Update all fields of the wedding plan state."""

    return Command(
        update={
            "name": name,
            "setting": setting,
            "location": location,
            "description": description,
            "persons": persons,
            "price": price,
            "messages": [
                ToolMessage(
                    content="Successfully updated the wedding plan details.",
                    tool_call_id=runtime.tool_call_id  # ← links back to the LLM tool call
                )
            ]
        }
    )


supervisor = create_agent(
    model='gpt-5-nano',
    system_prompt="You corrdinate wedding planning. Keep all informations about weeding in WeddingState, you can search net and call different agents \
        travel, dj, chief and venue. At the end return updated Weeding state. Note that cheif agent can plan one variation of menu at the time.",
    tools=[update_wedding_plan, web_search, call_travel_agent, call_dj_agent, call_chief_agent, call_venue_agent],
    checkpointer=InMemorySaver(),
    response_format=WeddingState
)



In [41]:
config = {'configurable': {"thread_id":"2"}}

message = 'I am getting merried with the love of my life. I want a small party for my closest friends and family (20 people). We love to keep it simple, but unforgetable. We live in Poland '
response = supervisor.invoke({'messages':[HumanMessage(content=message)]}, config=config)

In [50]:
def print_wedding_state(state: dict) -> None:
    """Print the wedding plan state in a pretty, readable format."""
    
    print("\n" + "="*55)
    print("           💍 WEDDING PLAN SUMMARY 💍")
    print("="*55)
    print(f"  📋 Name:        {state.get('name', 'N/A')}")
    print(f"  🏛️  Setting:     {state.get('setting', 'N/A')}")
    print(f"  📍 Location:    {state.get('location', 'N/A')}")
    print(f"  👥 Guests:      {state.get('persons', 'N/A')} people")
    print(f"  💰 Total Price: ${state.get('price', 0):,.2f}")
    
    # Description
    print("-"*55)
    print("  📝 Description:")
    description = state.get('description', 'N/A')
    # Word-wrap description at 50 chars
    words = description.split()
    line = "     "
    for word in words:
        if len(line) + len(word) + 1 > 55:
            print(line)
            line = "     " + word
        else:
            line += (" " if line != "     " else "") + word
    if line.strip():
        print(line)

    # Structured response summary if present
    structured = state.get('structured_response', {})
    if structured and isinstance(structured, dict):
        summary = structured.get('summary')
        if summary:
            print("-"*55)
            print("  🤖 AI Summary:")
            words = summary.split()
            line = "     "
            for word in words:
                if len(line) + len(word) + 1 > 55:
                    print(line)
                    line = "     " + word
                else:
                    line += (" " if line != "     " else "") + word
            if line.strip():
                print(line)

    print("="*55 + "\n")

In [51]:
print_wedding_state(response['structured_response'])


           💍 WEDDING PLAN SUMMARY 💍
  📋 Name:        Intimate Poland Wedding
  🏛️  Setting:     Minimalist, intimate, elegant
  📍 Location:    Poland (recommended: Goldwasser - Gdańsk; other options in Warsaw, Kraków, Wrocław)
  👥 Guests:      20 people
  💰 Total Price: $3,600.00
-------------------------------------------------------
  📝 Description:
     Small 20-person celebration in Poland with a
     minimalist, elegant vibe. Proposed venue:
     Goldwasser in Gdańsk for an intimate private
     dining space. Menu: Polish Harvest three-course
     menu with starter, main (with vegetarian option:
     mushroom golabki), and dessert; price approx. 180
     per person (total ~3600). DJ plan: intimate
     background music during dinner transitioning to
     tasteful dancing. Next steps: confirm venue and
     finalize dates and details.
-------------------------------------------------------
  🤖 AI Summary:
     Identified intimate, 20-person options in Poland;
     proposed Polish-

In [42]:
response['messages'][-1].content

'{"messages":[],"jump_to":null,"structured_response":{"summary":"Identified intimate, 20-person options in Poland; proposed Polish-inspired three-course menu with vegetarian option; DJ plan prepared; next steps: confirm venue (e.g., Goldwasser in Gdańsk) and finalize menu details."},"name":"Intimate Poland Wedding","setting":"Minimalist, intimate, elegant","location":"Poland (recommended: Goldwasser - Gdańsk; other options in Warsaw, Kraków, Wrocław)","description":"Small 20-person celebration in Poland with a minimalist, elegant vibe. Proposed venue: Goldwasser in Gdańsk for an intimate private dining space. Menu: Polish Harvest three-course menu with starter, main (with vegetarian option: mushroom golabki), and dessert; price approx. 180 per person (total ~3600). DJ plan: intimate background music during dinner transitioning to tasteful dancing. Next steps: confirm venue and finalize dates and details.","persons":20,"price":3600}'

In [52]:
response

{'messages': [HumanMessage(content='I am getting merried with the love of my life. I want a small party for my closest friends and family (20 people). We love to keep it simple, but unforgetable. We live in Poland ', additional_kwargs={}, response_metadata={}, id='2fdbf5a1-1162-4c24-b605-dafa9cb4012e'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2327, 'prompt_tokens': 519, 'total_tokens': 2846, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2240, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DgDDExXjc4hwqdKRvnNEuWnmNLFAl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e31db-98ce-7c81-bb7c-ea453c97df29-0', tool_calls=[{'name': 'update_w

For more usability supervisor agent should keep and return informations about playlist, menu and flights.
For full view of systems performance check: [trace](https://eu.smith.langchain.com/public/585132ff-74cf-417f-8916-6d65b3acd096/r).

This is my first experimental agent system, the idea is from langchain course.